# One-way ANOVA: Model and Example

In [4]:
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
import numpy as np

## Model

One-way ANOVA is used to test whether the means of several groups (treatments) are equal.

$$
Y_{ij} = \mu_i + \epsilon_{ij}
$$

Or equivalently,

$$
Y_{ij} = \mu + \tau_i + \epsilon_{ij}
$$

Where:
- $Y_{ij}$: the response from the $j$-th subject in group $i$  
- $\mu$: the overall mean
- $\mu_i$: the mean of treatment $i$  
- $\tau_i$: the effect of treatment $i$  
- $\epsilon_{ij} \sim N(0, \sigma^2)$: random error

## Example: Drug Dosage and Memory

Suppose a pharmaceutical company tests how different drug dosages affect memory scores.  
There are three treatment groups:
- Placebo (0 mg)
- Low dose (50 mg)
- High dose (100 mg)

Each group has 10 participants, and each participant takes a memory test.

### For example:
- $Y_{2,7}$: The response of the 7th subject in the 2nd treatment group (low dose)
- $\mu$: The overall mean memory score across all groups
- $\tau_2$: The treatment effect of the 2nd group (low dose), i.e., how much the low-dose group’s mean deviates from the overall mean
- $\epsilon_{2,7}$: The random error (or individual deviation) for subject 7 in group 2

### Hypotheses for One-way ANOVA

We test whether the means of all treatment groups are equal using the following hypotheses:

**Null Hypothesis ($H_0$):**  
All treatment effects are zero (no difference in group means)

$$
H_0: \mu_1 = \mu_2 = \cdots = \mu_v
$$

Equivalently:

$$
H_0: \tau_1 = \tau_2 = \cdots = \tau_v = 0
$$

**Notation**: the number of treatment groups is often denoted as $v$. In the case of our drug study, $v=3$.

**Alternative Hypothesis ($H_1$):**  
At least one group mean is different

$$
H_1: \text{Not all } \mu_i \text{ are equal.}
$$

or

$$
H_1: \text{At least one } \tau_i \neq 0
$$

### Reduced Model

If $H_0$ is true, we have a reduced model. 
$$
Y_{ij} = \mu + \epsilon_{ij}
$$


When assuming the reduced model (i.e., all treatment groups share the same mean), the sum of squares for error is:

$$
SSW = \sum_i \sum_j \left(Y_{ij} - \bar{Y}_{..}\right)^2
$$

- $Y_{ij}$: The observation from subject $j$ in group $i$
- $\bar{Y}_{..}$: The grand mean (overall average across all data)

This measures how far each individual observation is from the overall mean, ignoring group differences.

Under the null hypothesis (that all group means are equal), we estimate the variance of the error using:

$$
\frac{SSW}{n - 1}
$$

This is an **unbiased estimator** of the true error variance $\sigma^2$ when the null hypothesis is true.

### Full Model

Under the full model (where group means $\bar{Y}_i$ are allowed to differ),  
the sum of squares error (SSW) is computed as:

$$
SSW = \sum_i \sum_j (Y_{ij} - \bar{Y}_i)^2 = \sum_i (r_i - 1) S_i^2
$$

- $Y_{ij}$: Observation $j$ in group $i$
- $\bar{Y}_i$: Mean of group $i$
- $r_i$: Number of observations in group $i$
- $S_i^2$: Sample variance of group $i$

This measures the variability **within groups**, relative to their own group means.

To estimate the true error variance $\sigma^2$, we divide by the correct degrees of freedom:

$$
\frac{SSE}{n - v}
$$

- $n$: Total number of observations
- $v$: Number of groups

This is an **unbiased estimator** of $\sigma^2$, even when the group means are not equal.

### 📊 Decomposing Variability in One-way ANOVA

In one-way ANOVA, we decompose the total variability (SST) into:

- **Between-group variability (SSB)**: Variability explained by group differences
- **Within-group variability (SSW)**: Variability within each group

### 📌 Key Equation:

$$
SST = SSB + SSW
$$

### 🔍 From Sums of Squares:

$$
SST = \sum_i \sum_t (Y_{it} - \bar{Y}_{..})^2
$$

$$
SSW = \sum_i \sum_t (Y_{it} - \bar{Y}_i)^2
$$

Then:

$$
SSB = SST - SSW
$$

### 🔁 SSB Simplified:

$$
SSB = \sum_i r_i (\bar{Y}_i - \bar{Y}_{..})^2
$$

Or algebraically:

$$
SSB = \sum_i r_i (\bar{Y}_i)^2 - n (\bar{Y}_{..})^2
$$

- $r_i$: Number of observations in group $i$
- $\bar{Y}_i$: Group $i$ mean
- $\bar{Y}_{..}$: Grand mean
- $n$: Total number of observations

This decomposition lets us test whether **group means differ significantly**.

In [2]:
# 데이터 입력
data = {
    "driver": [1]*4 + [2]*4 + [3]*4,
    "mpg": [
        50.33, 46.83, 51.57, 45.33, 48.11,
        50.14, 43.22, 47.26, 49.08, 48.89,
        49.96, 49.70
    ]
}


# DataFrame 생성
df = pd.DataFrame(data)

# 확인
print(df)

# One-way ANOVA (driver 간 mpg 비교)
model = smf.ols('mpg ~ C(driver)', data=df).fit()
anova_table = sm.stats.anova_lm(model, typ=2)

print("\nANOVA Table:")
print(anova_table)

    driver    mpg
0        1  50.33
1        1  46.83
2        1  51.57
3        1  45.33
4        2  48.11
5        2  50.14
6        2  43.22
7        2  47.26
8        3  49.08
9        3  48.89
10       3  49.96
11       3  49.70

ANOVA Table:
              sum_sq   df         F    PR(>F)
C(driver)  10.030317  2.0  0.873196  0.450204
Residual   51.691050  9.0       NaN       NaN


In [5]:
# 처리 수준 (RPM)
x = [50, 75, 100, 125, 150]

# 각 처리 수준에 대한 샘플 수 (r_i)
r = [5, 3, 5, 2, 5]

# 각 처리 수준에 대한 반응값 평균 (μ_i)
mu = [20, 28, 38, 52, 60]  # 예시값

# 처리 수준 (x)와 샘플 수 (r)
x = np.array([50, 75, 100, 125, 150])
r = np.array([5, 3, 5, 2, 5])
mu = np.array([20, 28, 38, 52, 60])  # 처리 평균

# 전체 평균 x̄
x_bar = np.sum(r * x) / np.sum(r)

# contrast 계수 c_i = r_i * (x_i - x̄)
contrast_coeffs = r * (x - x_bar)

print("Contrast coefficients:", contrast_coeffs)

# 선형 contrast 값 계산: sum(c_i * mu_i)
linear_contrast = np.sum(contrast_coeffs * mu)

print("Linear trend contrast:", linear_contrast)

Contrast coefficients: [-243.75  -71.25    6.25   52.5   256.25]
Linear trend contrast: 11472.5


In [11]:
# 처리 수준 (x)와 샘플 수 (r)
x = np.array([50, 75, 100, 125, 150])
r = np.array([5, 3, 5, 2, 5])
mu = np.array([20, 28, 38, 52, 60])  # 처리 평균

# 전체 평균 x̄
x_bar = np.sum(r * x) / np.sum(r)

# contrast 계수 c_i = r_i * (x_i - x̄)
contrast_coeffs = r * (x - x_bar)

print("Contrast coefficients:", contrast_coeffs)

# 선형 contrast 값 계산: sum(c_i * mu_i)
linear_contrast = np.sum(contrast_coeffs * mu)

print("Linear trend contrast:", linear_contrast)

Contrast coefficients: [-243.75  -71.25    6.25   52.5   256.25]
Linear trend contrast: 11472.5


98.75